In [ ]:
import sys
import subprocess

required = ["transformers", "datasets", "scipy", "pandas", "torch"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import pipeline
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "cross-encoder/stsb-distilroberta-base"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


In [ ]:
pipe_device = "mps" if device == "mps" else -1

clf = pipeline(
    task="text-classification",
    model=model_name,
    tokenizer=model_name,
    device=pipe_device,
    return_all_scores=False,
)

print({
    "pipeline_task": clf.task,
    "model_name": model_name,
    "pipeline_device": pipe_device,
})


In [ ]:
pairs = [
    {"text": s1, "text_pair": s2}
    for s1, s2 in zip(df["sentence1"].tolist(), df["sentence2"].tolist())
]

outputs = clf(
    pairs,
    batch_size=batch_size,
    truncation=True,
)

raw_scores = np.array([float(x["score"]) for x in outputs], dtype=np.float32)
labels_out = [str(x.get("label", "")) for x in outputs]

def parse_label_value(label):
    text = str(label).strip().upper()
    for prefix in ["LABEL_", "STAR", "SCORE_"]:
        if text.startswith(prefix):
            text = text[len(prefix):]
            break
    try:
        return float(text)
    except Exception:
        return np.nan

label_values = np.array([parse_label_value(x) for x in labels_out], dtype=np.float32)

predicted_score_0_5 = np.where(np.isfinite(label_values), label_values, raw_scores)
predicted_score_0_5 = np.clip(predicted_score_0_5, 0.0, 5.0).astype(np.float32)

if np.nanmax(raw_scores) <= 1.0 + 1e-6:
    rescaled_from_probability = (raw_scores * 5.0).astype(np.float32)
else:
    rescaled_from_probability = np.clip(raw_scores, 0.0, 5.0).astype(np.float32)

print(pd.DataFrame({
    "label_name": labels_out[:10],
    "raw_score": raw_scores[:10],
    "predicted_score_0_5": predicted_score_0_5[:10],
    "rescaled_from_raw": rescaled_from_probability[:10],
} ))


In [ ]:
labels = df["label"].to_numpy(dtype=np.float32)

pearson_raw = pearsonr(raw_scores, labels).statistic
spearman_raw = spearmanr(raw_scores, labels).statistic

pearson_clipped = pearsonr(predicted_score_0_5, labels).statistic
spearman_clipped = spearmanr(predicted_score_0_5, labels).statistic

pearson_rescaled = pearsonr(rescaled_from_probability, labels).statistic
spearman_rescaled = spearmanr(rescaled_from_probability, labels).statistic

results_df = df.copy()
results_df["model_label"] = labels_out
results_df["raw_model_score"] = raw_scores
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["rescaled_from_raw"] = rescaled_from_probability
results_df["abs_error_clipped"] = np.abs(results_df["predicted_score_0_5"] - results_df["label"])
results_df["abs_error_rescaled"] = np.abs(results_df["rescaled_from_raw"] - results_df["label"])

print(results_df[[
    "sentence1", "sentence2", "label", "model_label", "raw_model_score",
    "predicted_score_0_5", "rescaled_from_raw", "abs_error_clipped"
]].head(10))


In [ ]:
error_table = results_df[[
    "sentence1", "sentence2", "label", "predicted_score_0_5", "rescaled_from_raw",
    "abs_error_clipped", "abs_error_rescaled"
]].copy()

print("worst_examples_by_clipped_error")
print(error_table.sort_values("abs_error_clipped", ascending=False).head(10).to_string(index=False))

runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"pearson_raw_model_score: {pearson_raw:.6f}")
print(f"spearman_raw_model_score: {spearman_raw:.6f}")
print(f"pearson_clipped_prediction: {pearson_clipped:.6f}")
print(f"spearman_clipped_prediction: {spearman_clipped:.6f}")
print(f"pearson_rescaled_from_raw: {pearson_rescaled:.6f}")
print(f"spearman_rescaled_from_raw: {spearman_rescaled:.6f}")
print(f"mae_clipped_prediction: {results_df['abs_error_clipped'].mean():.6f}")
print(f"mae_rescaled_from_raw: {results_df['abs_error_rescaled'].mean():.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
